# CASDA v5.6 파이프라인 — Colab 실행 노트북

Severstal Steel Defect Detection 프로젝트의 CASDA (Context-Aware Steel Defect Augmentation) v5.6 파이프라인.

## 파이프라인 구조 (11단계)

| Stage | Steps | 설명 | 리소스 |
|-------|-------|------|--------|
| A | 1-3 | 데이터 전처리 | CPU |
| B | 4-6 | ControlNet 학습 + 생성 | GPU |
| C | 7-9 | 후처리 + 품질 관리 | CPU |
| D | 10-11 | 평가 | GPU |

## v5.5 대비 개선
- 11단계 완전 파이프라인 (v5.5는 6단계)
- 실측 품질 점수 기반 pruning (v5.5는 점수 없이 top-k만)
- compositions-per-roi=3 (pruning pool 3배 확대)
- 전 스크립트 --workers -1 병렬화

## 제외된 스크립트
- `package_casda_data.py` — compose_casda_images.py가 generated/를 직접 처리하므로 불필요
- `merge_datasets.py` — run_benchmark.py 내부 자동 병합

---
## 0. 환경 설정

In [ ]:
# 0-1. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 0-2. 경로 설정 — 모든 셀에서 참조하는 핵심 변수
import os

# === 수정 필요: 본인의 Google Drive 경로에 맞게 설정 ===
BASE_DIR = "/content/drive/MyDrive/data/Severstal"
PROJECT_DIR = f"{BASE_DIR}/severstal_project"  # git clone 위치

# 원본 데이터
TRAIN_CSV     = f"{BASE_DIR}/train.csv"
TRAIN_IMAGES  = f"{BASE_DIR}/train_images"

# v5.5에서 재사용할 산출물 (Steps 1-6 건너뛸 때)
V55_DIR               = f"{BASE_DIR}/augmented_images_v5.5"
ROI_METADATA_CSV      = f"{BASE_DIR}/data/processed/roi_patches/roi_metadata.csv"
CONTROLNET_DATASET    = f"{BASE_DIR}/data/processed/controlnet_dataset"
CONTROLNET_BEST_MODEL = f"{V55_DIR}/best_model"      # 또는 학습 output_dir
TRAINING_LOG_JSON     = f"{V55_DIR}/training_log.json"
GENERATED_DIR         = f"{V55_DIR}/generated"
GENERATION_SUMMARY    = f"{V55_DIR}/generation_summary.json"
HINT_DIR              = f"{CONTROLNET_DATASET}/hints"
TRAIN_JSONL           = f"{CONTROLNET_DATASET}/train.jsonl"

# v5.6 출력 경로
V56_DIR           = f"{BASE_DIR}/outputs/v5.6"
BACKGROUNDS_DIR   = f"{BASE_DIR}/data/backgrounds"
CASDA_COMPOSED    = f"{V56_DIR}/casda_composed"
FID_OUTPUT        = f"{V56_DIR}/fid"
BENCHMARK_OUTPUT  = f"{V56_DIR}/benchmark"

# 설정 파일
CONFIG_YAML = f"{PROJECT_DIR}/configs/benchmark_experiment.yaml"

# v5.5 벤치마크 결과 (reference)
V55_BENCHMARK_JSON = f"{BASE_DIR}/outputs/v5.5/benchmark_results.json"

# 출력 디렉토리 생성
os.makedirs(V56_DIR, exist_ok=True)

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"V56_DIR:     {V56_DIR}")
print(f"TRAIN_CSV:   {TRAIN_CSV} (exists: {os.path.isfile(TRAIN_CSV)})")
print(f"TRAIN_IMAGES: {TRAIN_IMAGES} (exists: {os.path.isdir(TRAIN_IMAGES)})")

In [ ]:
# 0-3. 의존성 설치
!pip install -q diffusers transformers accelerate safetensors xformers \
    lpips albumentations ultralytics \
    scikit-image scipy seaborn tqdm PyYAML

In [ ]:
# 0-4. 프로젝트 루트를 sys.path에 추가
import sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# GPU 확인
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# 0-5. v5.5 산출물 존재 확인 (재사용 가능 여부 판단)
reusable = {
    "roi_metadata.csv":     os.path.isfile(ROI_METADATA_CSV),
    "controlnet_dataset/":  os.path.isdir(CONTROLNET_DATASET),
    "best_model/":          os.path.isdir(CONTROLNET_BEST_MODEL),
    "generated/":           os.path.isdir(GENERATED_DIR),
    "generation_summary":   os.path.isfile(GENERATION_SUMMARY),
    "hints/":               os.path.isdir(HINT_DIR),
    "train.jsonl":          os.path.isfile(TRAIN_JSONL),
    "backgrounds/":         os.path.isdir(BACKGROUNDS_DIR),
}

print("=" * 50)
print("v5.5 산출물 재사용 가능 여부")
print("=" * 50)
all_reusable = True
for name, exists in reusable.items():
    status = "OK" if exists else "MISSING"
    if not exists:
        all_reusable = False
    print(f"  {status:7s}  {name}")

print()
if all_reusable:
    print(">>> Steps 1-6 재사용 가능. Step 7부터 시작하세요.")
else:
    print(">>> 누락된 산출물이 있습니다. 해당 Step부터 실행하세요.")

---
## Stage A: 데이터 전처리 (CPU, 1회성)

v5.5 산출물이 존재하면 **Stage A 전체를 건너뛸 수 있습니다.**

### Step 1. ROI 추출 (`extract_rois.py`)

- **입력**: `train.csv`, `train_images/`
- **출력**: `roi_patches/roi_metadata.csv` (15,380 ROI)
- **재사용 조건**: `roi_metadata.csv` 존재 & roi_size=256 동일

In [ ]:
# Step 1: ROI 추출
# v5.5 산출물이 있으면 건너뛰기
SKIP_STEP_1 = os.path.isfile(ROI_METADATA_CSV)

if SKIP_STEP_1:
    print(f"SKIP: roi_metadata.csv 이미 존재 → {ROI_METADATA_CSV}")
else:
    !python {PROJECT_DIR}/scripts/extract_rois.py \
        --image_dir {TRAIN_IMAGES} \
        --train_csv {TRAIN_CSV} \
        --output_dir {os.path.dirname(ROI_METADATA_CSV)} \
        --roi_size 256 \
        --min_suitability 0.5 \
        --num_workers -1

### Step 2. 배경 추출 (`extract_clean_backgrounds.py`)

- **입력**: `train.csv`, `train_images/`
- **출력**: `backgrounds/` (512x512 클린 배경 패치)
- **재사용 조건**: `backgrounds/` 존재

In [ ]:
# Step 2: 배경 추출
SKIP_STEP_2 = os.path.isdir(BACKGROUNDS_DIR) and len(os.listdir(BACKGROUNDS_DIR)) > 100

if SKIP_STEP_2:
    print(f"SKIP: backgrounds/ 이미 존재 ({len(os.listdir(BACKGROUNDS_DIR))} files)")
else:
    !python {PROJECT_DIR}/scripts/extract_clean_backgrounds.py \
        --train_csv {TRAIN_CSV} \
        --image_dir {TRAIN_IMAGES} \
        --output_dir {BACKGROUNDS_DIR} \
        --patch_size 512 \
        --patches_per_image 5 \
        --min_quality 0.7 \
        --workers -1

### Step 3. ControlNet 학습 데이터 준비 (`prepare_controlnet_data.py`)

- **입력**: `roi_metadata.csv`, `train_images/`, `train.csv`
- **출력**: `controlnet_dataset/` (hints/, train.jsonl, metadata.json)
- **재사용 조건**: `controlnet_dataset/train.jsonl` 존재 & per_class_cap 동일

In [ ]:
# Step 3: ControlNet 학습 데이터 준비
SKIP_STEP_3 = os.path.isfile(TRAIN_JSONL)

if SKIP_STEP_3:
    print(f"SKIP: train.jsonl 이미 존재 → {TRAIN_JSONL}")
else:
    !python {PROJECT_DIR}/scripts/prepare_controlnet_data.py \
        --roi_metadata {ROI_METADATA_CSV} \
        --train_images {TRAIN_IMAGES} \
        --train_csv {TRAIN_CSV} \
        --output_dir {CONTROLNET_DATASET} \
        --prompt_style detailed \
        --per_class_cap 900 \
        --rare_class_threshold 200 \
        --workers -1

---
## Stage B: ControlNet 학습 + 생성 (GPU)

v5.5 모델(`best_model/`)과 생성 결과(`generated/`)가 존재하면 **Stage B 전체를 건너뛸 수 있습니다.**

### Step 4. ControlNet 학습 (`train_controlnet.py`)

- **입력**: `controlnet_dataset/`
- **출력**: `best_model/`, `training_log.json`
- **재사용 조건**: `best_model/` 존재
- **소요 시간**: T4 기준 약 4-8시간

In [ ]:
# Step 4: ControlNet 학습
SKIP_STEP_4 = os.path.isdir(CONTROLNET_BEST_MODEL)

if SKIP_STEP_4:
    print(f"SKIP: best_model/ 이미 존재 → {CONTROLNET_BEST_MODEL}")
else:
    TRAIN_OUTPUT_DIR = f"{V56_DIR}/controlnet_training"
    !python {PROJECT_DIR}/scripts/train_controlnet.py \
        --data_dir {CONTROLNET_DATASET} \
        --image_root {TRAIN_IMAGES} \
        --resolution 512 \
        --train_batch_size 1 \
        --num_train_epochs 100 \
        --gradient_accumulation_steps 4 \
        --gradient_checkpointing \
        --mixed_precision fp16 \
        --force_grayscale_target \
        --snr_gamma 5.0 \
        --early_stopping_patience 20 \
        --learning_rate 1e-5 \
        --lr_scheduler cosine \
        --lr_warmup_steps 50 \
        --output_dir {TRAIN_OUTPUT_DIR} \
        --validation_steps 200 \
        --checkpointing_steps 500 \
        --checkpoints_total_limit 3 \
        --save_fp16 \
        --skip_save_pipeline \
        --seed 42
    
    # 학습 완료 후 best_model 경로 갱신
    CONTROLNET_BEST_MODEL = f"{TRAIN_OUTPUT_DIR}/best_model"
    TRAINING_LOG_JSON = f"{TRAIN_OUTPUT_DIR}/training_log.json"
    print(f"학습 완료: {CONTROLNET_BEST_MODEL}")

### Step 5. ControlNet 모델 검증 (`run_validation_phases.py`) [선택]

- **입력**: `best_model/`, `train.jsonl`
- **출력**: `validation/` (검증 보고서, 비교 그리드)
- **참고**: 선택적 단계. 모델 품질 확인용.

In [ ]:
# Step 5: ControlNet 검증 (선택)
RUN_STEP_5 = False  # True로 변경하면 실행

if RUN_STEP_5:
    VALIDATION_OUTPUT = f"{V56_DIR}/validation"
    !python {PROJECT_DIR}/scripts/run_validation_phases.py \
        --model_path {CONTROLNET_BEST_MODEL} \
        --jsonl_path {TRAIN_JSONL} \
        --image_root {TRAIN_IMAGES} \
        --roi_metadata_path {ROI_METADATA_CSV} \
        --training_log_path {TRAINING_LOG_JSON} \
        --output_base {VALIDATION_OUTPUT} \
        --phases 1 2 3 4 \
        --quality_threshold 0.5 \
        --controlnet_conditioning_scale 0.7 \
        --workers -1
else:
    print("SKIP: Step 5 (선택적 검증 단계)")

### Step 6. 합성 이미지 생성 (`test_controlnet.py`)

- **입력**: `best_model/`, `train.jsonl`
- **출력**: `generated/`, `generation_summary.json`
- **재사용 조건**: `generated/` 존재 & `generation_summary.json` 존재
- **소요 시간**: T4 기준 약 1-3시간 (배치 크기에 따라)

In [ ]:
# Step 6: 합성 이미지 생성
SKIP_STEP_6 = os.path.isdir(GENERATED_DIR) and os.path.isfile(GENERATION_SUMMARY)

if SKIP_STEP_6:
    n_gen = len([f for f in os.listdir(GENERATED_DIR) if f.endswith('.png')])
    print(f"SKIP: generated/ 이미 존재 ({n_gen} images)")
else:
    GENERATED_DIR = f"{V56_DIR}/generated"
    !python {PROJECT_DIR}/scripts/test_controlnet.py \
        --model_path {CONTROLNET_BEST_MODEL} \
        --jsonl_path {TRAIN_JSONL} \
        --image_root {TRAIN_IMAGES} \
        --num_inference_steps 30 \
        --guidance_scale 7.5 \
        --controlnet_conditioning_scale 0.7 \
        --num_images_per_sample 1 \
        --resolution 512 \
        --grayscale_postprocess \
        --output_dir {GENERATED_DIR} \
        --seed 42
    
    GENERATION_SUMMARY = f"{GENERATED_DIR}/generation_summary.json"
    print(f"생성 완료: {GENERATED_DIR}")

---
## Stage C: 후처리 + 품질 관리 (CPU)

**v5.6 핵심 단계.** v5.5에서 누락되었던 compose → score → validate를 실행합니다.

`compose_casda_images.py`가 `generated/`와 `hints/`를 직접 받아서 1600x256 합성 이미지를 생성합니다.
(`package_casda_data.py`는 불필요 — compose가 마스크 추출·합성을 자체 처리)

### Step 7. Poisson Blending 합성 (`compose_casda_images.py`)

- **입력**: `generated/`, `hints/`, `roi_metadata.csv`, `generation_summary.json`, `train_images/`, `train.csv`
- **출력**: `casda_composed/` (1600x256 합성 이미지, metadata.json)
- **핵심 변경**: `--compositions-per-roi 3` (pruning pool 3배 확대)
- **소요 시간**: CPU 병렬 기준 약 30-60분

In [ ]:
# Step 7: Poisson Blending 합성
# compose_casda_images.py가 generated/ + hints/를 직접 처리
# (package_casda_data.py 중간 단계 불필요)

!python {PROJECT_DIR}/scripts/compose_casda_images.py \
    --generated-dir {GENERATED_DIR} \
    --hint-dir {HINT_DIR} \
    --metadata-csv {ROI_METADATA_CSV} \
    --summary-json {GENERATION_SUMMARY} \
    --clean-images-dir {TRAIN_IMAGES} \
    --train-csv {TRAIN_CSV} \
    --output-dir {CASDA_COMPOSED} \
    --dilation-px 8 \
    --blend-mode NORMAL_CLONE \
    --jitter-range 100 \
    --scale-min 0.875 \
    --scale-max 1.0 \
    --smooth-mask \
    --smooth-ksize 21 \
    --smooth-sigma 7.0 \
    --brightness-tolerance 30.0 \
    --compositions-per-roi 3 \
    --png-compression 1 \
    --seed 42 \
    --workers -1

# 결과 확인
if os.path.isdir(CASDA_COMPOSED):
    composed_images = f"{CASDA_COMPOSED}/images"
    if os.path.isdir(composed_images):
        n = len(os.listdir(composed_images))
        print(f"합성 완료: {n} images in {composed_images}")
    metadata_path = f"{CASDA_COMPOSED}/metadata.json"
    print(f"metadata.json exists: {os.path.isfile(metadata_path)}")

### Step 8. 합성 품질 점수 산출 (`score_casda_quality.py`)

- **입력**: `casda_composed/` (metadata.json + images/)
- **출력**: `metadata.json` 내 `quality_score` 필드 갱신
- **메트릭**: color_score (0.4) + artifact_score (0.3) + blur_score (0.3)

In [ ]:
# Step 8: 품질 점수 산출
!python {PROJECT_DIR}/scripts/score_casda_quality.py \
    --casda-dir {CASDA_COMPOSED} \
    --weight-color 0.40 \
    --weight-artifact 0.30 \
    --weight-blur 0.30 \
    --batch-log-interval 500 \
    --workers -1

# 점수 분포 확인
import json
metadata_path = f"{CASDA_COMPOSED}/metadata.json"
if os.path.isfile(metadata_path):
    with open(metadata_path) as f:
        meta = json.load(f)
    scores = [s.get('quality_score', 0) for s in meta.get('samples', [])]
    if scores:
        import numpy as np
        scores = np.array(scores)
        print(f"Quality Score 분포:")
        print(f"  N={len(scores)}, mean={scores.mean():.4f}, std={scores.std():.4f}")
        print(f"  min={scores.min():.4f}, P25={np.percentile(scores, 25):.4f}, ")
        print(f"  P50={np.percentile(scores, 50):.4f}, P75={np.percentile(scores, 75):.4f}, max={scores.max():.4f}")

### Step 9. 품질 검증 (`validate_augmented_quality.py`)

- **입력**: `casda_composed/`
- **출력**: `validation/` (quality_scores.json, validation_report.json)
- **판정 기준**: `--min_quality_score 0.7`

In [ ]:
# Step 9: 품질 검증
VALIDATION_DIR = f"{V56_DIR}/quality_validation"

!python {PROJECT_DIR}/scripts/validate_augmented_quality.py \
    --augmented_dir {CASDA_COMPOSED} \
    --output_dir {VALIDATION_DIR} \
    --min_quality_score 0.7 \
    --workers -1

# 검증 결과 확인
report_path = f"{VALIDATION_DIR}/validation_report.json"
if os.path.isfile(report_path):
    with open(report_path) as f:
        report = json.load(f)
    print("검증 결과:")
    print(json.dumps(report, indent=2, ensure_ascii=False)[:2000])

---
## Stage D: 평가 (GPU)

FID 평가와 모델 학습+벤치마크를 수행합니다.

### Step 10. FID 평가 (`run_fid.py`)

- **입력**: `train_images/`, `casda_composed/`, `generated/`, `roi_metadata.csv`
- **출력**: `fid/fid_results.json`
- **모드**: `both` (FID-ROI + FID-Composed)

In [ ]:
# Step 10: FID 평가
!python {PROJECT_DIR}/scripts/run_fid.py \
    --config {CONFIG_YAML} \
    --data-dir {TRAIN_IMAGES} \
    --csv {TRAIN_CSV} \
    --casda-dir {CASDA_COMPOSED} \
    --casda-roi-dir {GENERATED_DIR} \
    --output-dir {FID_OUTPUT} \
    --fid-mode both \
    --device cuda

# FID 결과 확인
fid_result_path = f"{FID_OUTPUT}/fid_results.json"
if os.path.isfile(fid_result_path):
    with open(fid_result_path) as f:
        fid_data = json.load(f)
    print("FID Results:")
    print(json.dumps(fid_data, indent=2)[:3000])

### Step 11. 모델 학습 + 벤치마크 (`run_benchmark.py`)

- **입력**: config, 모든 데이터 경로
- **출력**: `benchmark/benchmark_results.json`
- **실행**: 3 models x 4 dataset groups = 12 training runs
- **소요 시간**: T4 기준 약 12-24시간

**참고**: `run_benchmark.py`는 내부에서 CASDA 데이터를 자동 병합합니다.
- Detection (YOLO): `inject_casda_to_baseline()` → symlink 주입
- Segmentation (DeepLabV3+): `ConcatDataset`으로 결합

In [ ]:
# Step 11-A: 벤치마크 실행 (전체 4그룹)
# 장시간 소요 — T4 기준 12-24시간
# 중단 후 --resume로 재개 가능

!python {PROJECT_DIR}/scripts/run_benchmark.py \
    --config {CONFIG_YAML} \
    --data-dir {TRAIN_IMAGES} \
    --csv {TRAIN_CSV} \
    --casda-dir {CASDA_COMPOSED} \
    --casda-roi-dir {GENERATED_DIR} \
    --output-dir {BENCHMARK_OUTPUT} \
    --groups all \
    --device cuda \
    --seed 42

In [ ]:
# Step 11-B: 벤치마크 중단 후 재개 (필요 시)
# --resume 플래그로 완료된 실험 건너뜀

# !python {PROJECT_DIR}/scripts/run_benchmark.py \
#     --config {CONFIG_YAML} \
#     --data-dir {TRAIN_IMAGES} \
#     --csv {TRAIN_CSV} \
#     --casda-dir {CASDA_COMPOSED} \
#     --casda-roi-dir {GENERATED_DIR} \
#     --output-dir {BENCHMARK_OUTPUT} \
#     --groups all \
#     --device cuda \
#     --seed 42 \
#     --resume

In [ ]:
# Step 11-C: v5.5 결과와 비교 (reference-results)

# !python {PROJECT_DIR}/scripts/run_benchmark.py \
#     --config {CONFIG_YAML} \
#     --data-dir {TRAIN_IMAGES} \
#     --csv {TRAIN_CSV} \
#     --casda-dir {CASDA_COMPOSED} \
#     --casda-roi-dir {GENERATED_DIR} \
#     --output-dir {BENCHMARK_OUTPUT} \
#     --groups all \
#     --device cuda \
#     --seed 42 \
#     --resume \
#     --reference-results {V55_BENCHMARK_JSON}

---
## 벤치마크 결과 확인

In [ ]:
# 벤치마크 결과 로드 및 요약
import json
import pandas as pd

benchmark_path = f"{BENCHMARK_OUTPUT}/benchmark_results.json"
if os.path.isfile(benchmark_path):
    with open(benchmark_path) as f:
        results = json.load(f)
    
    print("=" * 70)
    print("CASDA v5.6 벤치마크 결과")
    print("=" * 70)
    
    rows = []
    for run in results.get('runs', []):
        model = run.get('model', '')
        group = run.get('group', '')
        metrics = run.get('metrics', {})
        row = {'Model': model, 'Group': group}
        row.update(metrics)
        rows.append(row)
    
    if rows:
        df = pd.DataFrame(rows)
        print(df.to_string(index=False))
else:
    print(f"결과 파일 없음: {benchmark_path}")

---
## P3: Ratio 실험 (10% / 20% / 30% / 50%)

CASDA 합성 데이터 주입 비율별 성능 비교. 위 벤치마크 완료 후 실행.

In [ ]:
# P3: Ratio 실험
# --casda-ratio로 여러 비율을 한 번에 실험
# --casda-ratio-source composed: casda_composed 디렉토리 사용

RATIO_OUTPUT = f"{V56_DIR}/ratio_experiment"

!python {PROJECT_DIR}/scripts/run_benchmark.py \
    --config {CONFIG_YAML} \
    --data-dir {TRAIN_IMAGES} \
    --csv {TRAIN_CSV} \
    --casda-dir {CASDA_COMPOSED} \
    --casda-roi-dir {GENERATED_DIR} \
    --output-dir {RATIO_OUTPUT} \
    --casda-ratio 0.10 0.20 0.30 0.50 \
    --casda-ratio-source composed \
    --device cuda \
    --seed 42

In [ ]:
# Ratio 실험 결과 확인
ratio_path = f"{RATIO_OUTPUT}/benchmark_results.json"
if os.path.isfile(ratio_path):
    with open(ratio_path) as f:
        ratio_results = json.load(f)
    
    print("=" * 70)
    print("CASDA Ratio 실험 결과")
    print("=" * 70)
    
    rows = []
    for run in ratio_results.get('runs', []):
        model = run.get('model', '')
        group = run.get('group', '')
        metrics = run.get('metrics', {})
        row = {'Model': model, 'Group': group}
        row.update(metrics)
        rows.append(row)
    
    if rows:
        df = pd.DataFrame(rows)
        print(df.to_string(index=False))
else:
    print(f"결과 파일 없음: {ratio_path}")

---
## 세션 정리

Colab 세션 종료 전 중요 산출물 위치 확인:

In [ ]:
# 산출물 요약
print("=" * 70)
print("v5.6 산출물 디렉토리")
print("=" * 70)

outputs = {
    "casda_composed/":     CASDA_COMPOSED,
    "fid/":                FID_OUTPUT,
    "benchmark/":          BENCHMARK_OUTPUT,
    "quality_validation/": f"{V56_DIR}/quality_validation",
}

for name, path in outputs.items():
    exists = os.path.isdir(path)
    if exists:
        n_files = sum(1 for _ in os.scandir(path) if _.is_file())
        n_dirs = sum(1 for _ in os.scandir(path) if _.is_dir())
        print(f"  {name:25s} {path}")
        print(f"  {'':25s} ({n_files} files, {n_dirs} subdirs)")
    else:
        print(f"  {name:25s} NOT FOUND")

print()
print("Google Drive에 자동 동기화됩니다. Colab 세션을 종료해도 데이터는 보존됩니다.")